## Practice with Zarr

In [58]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('diffusionsim')))
import diffusers
import diffusionsim as diff
import diffusionsim.training_utils as tru
from diffusionsim import mydatasets as data
from diffusionsim import climsim_utils as cut
path = lambda fname : os.path.join(os.path.expanduser("~/diffusion-climsim/"), fname)

In [59]:
from virtualizarr import open_virtual_dataset
import zarr
import kerchunk
import icechunk
import xarray as xr
import fsspec

import json
import numpy as np

In [7]:
dconfig = tru.DataConfig()
dconfig.source = "local"
dconfig.climsim_type = "low-res-expanded"
dconfig.data_dir = "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/train"


In [132]:
print(os.listdir(dconfig.data_dir)[0])

year, month = map(int, os.listdir(dconfig.data_dir)[0].split("-"))

os.listdir(dconfig.data_dir)

0004-05


['0004-05',
 '0005-12',
 '0007-05',
 '0002-06',
 '0005-03',
 '0005-06',
 '0002-04',
 '0002-11',
 '0002-12',
 '0005-11',
 '0007-01',
 '0003-08',
 '0005-09',
 '0008-08',
 '0008-10',
 '0006-05',
 '0005-02',
 '0004-07',
 '0006-02',
 '0006-08',
 '0004-01',
 '0008-02',
 '0007-09',
 '0001-11',
 '0006-07',
 '0006-09',
 '0004-04',
 '0001-08',
 '0005-08',
 '0007-03',
 '0008-06',
 '0005-10',
 '0008-04',
 '0002-07',
 '0004-10',
 '0001-07',
 '0003-04',
 '0001-10',
 '0003-03',
 '0007-07',
 '0004-12',
 '0008-07',
 '0003-06',
 '0001-09',
 '0008-03',
 '0001-12',
 '0003-01',
 '0002-03',
 '0008-09',
 '0004-08',
 '0008-11',
 '0003-05',
 '0007-02',
 '0005-01',
 '0005-05',
 '0007-10',
 '0003-09',
 '0006-10',
 '0008-01',
 '0006-06',
 '0002-08',
 '0002-09',
 '0002-05',
 '0004-09',
 '0003-07',
 '0004-03',
 '0003-12',
 '0007-12',
 '0006-04',
 '0008-05',
 '0006-12',
 '0004-11',
 '0005-07',
 '0003-11',
 '0007-08',
 '0004-02',
 '0005-04',
 '0002-10',
 '0006-03',
 '0002-01',
 '0004-06',
 '0002-02',
 '0007-06',
 '00

In [9]:
kwargs = {
    'base_dir' : "/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/train",
}
kwargs['grid_info'] = xr.open_dataset(os.path.join(kwargs['base_dir'], "ClimSim_low-res_grid-info.nc"))

dutils = cut.setup_data_utils(dconfig.climsim_type, dconfig.source, dconfig.data_vars, use_tendencies=dconfig.use_tendencies, **kwargs)

start = cut.tocft(2, 1, 1)
stop = cut.tocft(2, 1, 15)

dutils.set_filelist_using_intervals("train", start, stop, 20*3*6)

In [124]:
d = json.load(open(path("combined.json"), 'r'))

d['refs']['ymd/0']

['/mnt/lustre/columbia/ssa2206/data/ClimSim_low-res-expanded/train/0002-01/E3SM-MMF.mlexpand.0002-01-01-00000.nc',
 1585,
 4]

In [131]:
start = cut.tocft(4, 6, 1)
stop = cut.tocft(4, 7, 1)
dutils.set_filelist_using_intervals("train", start, stop, 20*3*6)
dutils.get_filelist("train")

['0004-06/E3SM-MMF.mlexpand.0004-06-01-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-01-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-01-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-01-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-02-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-03-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-04-64800.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-00000.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-21600.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-43200.nc',
 '0004-06/E3SM-MMF.mlexpand.0004-06-05-64800.nc',


In [12]:
filelist = dutils.get_filelist("train")
len(filelist)

56

In [126]:
virtual_datasets = []
times = []
for i, fname in enumerate(filelist[:10]):
    fpath = os.path.join(dutils.data_path, fname)
    ds = open_virtual_dataset(fpath)
    time = dutils.parse_time(fname)
    times.append(time)
    ds = ds.expand_dims(time=[i+1])
    #ds["time"] = (["time"], [i])
    virtual_datasets.append(ds)


In [137]:
np.load(path("times.npy"), allow_pickle=True)

array([cftime.DatetimeNoLeap(2, 1, 1, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 1, 6, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 1, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 1, 18, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 6, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 12, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 2, 18, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 3, 0, 0, 0, 0, has_year_zero=True),
       cftime.DatetimeNoLeap(2, 1, 3, 6, 0, 0, 0, has_year_zero=True)],
      dtype=object)

In [135]:
np.save(path("times.npy"), times, allow_pickle=True)

In [86]:
virtual_ds = xr.combine_nested(virtual_datasets, concat_dim=['time'])
virtual_ds

<xarray.Dataset> Size: 60MB
Dimensions:                (time: 10, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) int64 80B 1 2 3 4 5 6 7 8 9 10
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    ymd                    (time) int32 40B ManifestArray<shape=(10,), dtype=...
    tod                    (time) int32 40B ManifestArray<shape=(10,), dtype=...
    cam_in_ALDIF           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    cam_in_ALDIR           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    cam_in_ASDIF           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    cam_in_ASDIR           (time, ncol) float64 31kB ManifestArray<shape=(10,...
    ...                     ...
    tm_pbuf_COSZRS         (time, ncol) float64 31kB ManifestArray<shape=(10,...
    lat                    (time, ncol) float64 31kB ManifestArray<shape=(10,...
    lon                    (time, ncol) float64 31kB ManifestArray<shape=(10,...
    clat                   (time, ncol) float64 31kB ManifestArray<shape=(10,...
    slat                   (time, ncol) float64 31kB ManifestArray<shape=(10,...
    icol                   (time, ncol) float64 31kB ManifestArray<shape=(10,...

In [88]:
#virtual_ds.virtualize.to_kerchunk(path("combined.json"), format='json')
virtual_ds.virtualize.to_kerchunk(path("combined2.parquet"), format='parquet')

In [89]:
combined_ds = xr.open_dataset(path('combined2.parquet'), engine="kerchunk", chunks={})

In [ ]:
ds = xr.open_dataset('combined.json', engine='kerchunk', chunks={}

In [96]:
dsi

<xarray.Dataset> Size: 4MB
Dimensions:      (time: 10, lev: 60, ncol: 384)
Coordinates:
  * time         (time) object 80B 0002-01-01 00:00:00 ... 0002-01-03 06:00:00
Dimensions without coordinates: lev, ncol
Data variables:
    state_t      (time, lev, ncol) float64 2MB -inf -inf inf ... -inf -inf -inf
    state_q0001  (time, lev, ncol) float64 2MB -inf -inf -inf ... -inf -inf -inf
    state_ps     (time, ncol) float64 31kB inf inf inf inf ... -inf -inf inf inf
    pbuf_SOLIN   (time, ncol) float64 31kB -inf -inf -inf -inf ... inf -inf -inf
    pbuf_LHFLX   (time, ncol) float64 31kB -inf -inf -inf -inf ... -inf inf inf
    pbuf_SHFLX   (time, ncol) float64 31kB -inf -inf -inf -inf ... inf inf inf

In [99]:
ds_inputs, ds_targets = [], []
for i, file in enumerate(filelist):
    ds_input = dutils.get_input(file)
    ds_target = dutils.get_target(file)
    ds_inputs.append(ds_input)
    ds_targets.append(ds_target)

ds_inputs = xr.concat(ds_inputs, dim='time')
ds_targets = xr.concat(ds_targets, dim='time')


In [104]:
ds_inputs

<xarray.Dataset> Size: 21MB
Dimensions:      (time: 56, lev: 60, ncol: 384)
Coordinates:
  * time         (time) object 448B 0002-01-01 00:00:00 ... 0002-01-14 18:00:00
Dimensions without coordinates: lev, ncol
Data variables:
    state_t      (time, lev, ncol) float64 10MB 212.6 209.0 ... 267.4 268.7
    state_q0001  (time, lev, ncol) float64 10MB 1.046e-06 1.046e-06 ... 0.00189
    state_ps     (time, ncol) float64 172kB 1.007e+05 1.016e+05 ... 1.004e+05
    pbuf_SOLIN   (time, ncol) float64 172kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    pbuf_LHFLX   (time, ncol) float64 172kB 41.66 31.69 53.02 ... 82.58 100.7
    pbuf_SHFLX   (time, ncol) float64 172kB 3.204 4.14 8.524 ... 93.91 86.44
Attributes:
    ne:        4
    fv_nphys:  2
    calendar:  NO_LEAP

In [109]:
ds_inputs['state_t'].data

array([[[212.5978174 , 208.96261498, 219.0162239 , ..., 216.18261307,
         217.87367276, 223.77638051],
        [218.3212459 , 216.91067342, 226.43448218, ..., 219.37087145,
         237.19641764, 237.76134192],
        [231.7245395 , 232.51944625, 232.54869655, ..., 231.19627076,
         238.42716583, 238.35975779],
        ...,
        [291.50802812, 287.98873335, 294.67627256, ..., 255.0645094 ,
         272.16110837, 272.37923022],
        [292.48501245, 289.10676856, 295.1791328 , ..., 256.02903722,
         272.9716271 , 273.52246283],
        [293.64563862, 290.28940923, 296.05201769, ..., 257.02499625,
         274.0153692 , 274.7273004 ]],

       [[210.7304669 , 210.0221052 , 215.13246685, ..., 219.19644466,
         224.72313981, 226.18673286],
        [218.4391942 , 226.19061597, 217.44592552, ..., 224.14801855,
         237.62795576, 232.95331796],
        [230.16199482, 234.45166202, 230.65427327, ..., 234.20125841,
         238.22917903, 239.75267657],
        ...,


In [112]:
for var in dutils.input_vars:
    print(var)
    print((ds_inputs[var].data[:10] == combined_ds[var].data).all())


state_t
True
state_q0001
True
state_ps
True
pbuf_SOLIN
True
pbuf_LHFLX
True
pbuf_SHFLX
True


In [110]:
combined_ds

<xarray.Dataset> Size: 60MB
Dimensions:                (time: 10, ncol: 384, lev: 60)
Coordinates:
  * time                   (time) float64 80B 1.0 2.0 3.0 4.0 ... 8.0 9.0 10.0
Dimensions without coordinates: ncol, lev
Data variables: (12/61)
    cam_in_ALDIF           (time, ncol) float64 31kB 1.0 1.0 ... 0.09448 0.09341
    cam_in_ALDIR           (time, ncol) float64 31kB 1.0 1.0 ... 0.3732 0.2052
    cam_in_ASDIF           (time, ncol) float64 31kB 1.0 1.0 ... 0.09298 0.06892
    cam_in_ASDIR           (time, ncol) float64 31kB 1.0 1.0 ... 0.3573 0.1778
    cam_in_ICEFRAC         (time, ncol) float64 31kB 0.0 0.0 0.0 ... 0.0 0.0
    cam_in_LANDFRAC        (time, ncol) float64 31kB 0.0 0.0 ... 0.1417 0.2428
    ...                     ...
    tm_state_u             (time, lev, ncol) float64 2MB -41.94 -58.75 ... 6.64
    tm_state_u_dyn         (time, lev, ncol) float64 2MB 0.001111 ... -0.0001621
    tm_state_u_prvphy      (time, lev, ncol) float64 2MB 0.0 0.0 ... 0.0001139
    tm_state_v             (time, lev, ncol) float64 2MB -9.322 ... -3.942
    tod                    (time) float64 80B 0.0 2.16e+04 ... 0.0 2.16e+04
    ymd                    (time) float64 80B 2.01e+04 2.01e+04 ... 2.01e+04

##### Debugging

In [ ]:
ref = json.load(open(os.path.expanduser("~/diffusion-climsim/combined.json"), 'r'))

### Working with actual zarrs

[Zarr Array Docs](https://zarr.readthedocs.io/en/latest/user-guide/arrays.html)

In [ ]:
mapper = fsspec.get_mapper("reference://", 
                           fo=path("combined.json"), 
                           target_protocol="file")



zgroup = zarr.open_group(mapper, mode="r+")

In [52]:
store = zarr.storage.MemoryStore()
z = zarr.create_array(store=store, shape=(1000, 1000), chunks=(100, 100), dtype='int32')
z

<Array memory://23451371698944 shape=(1000, 1000) dtype=int32>

In [56]:
z[0, :] = np.arange(1000)
z[:, 0] = np.arange(1000)

In [57]:
z[:]

array([[  0,   1,   2, ..., 997, 998, 999],
       [  1,   0,   0, ...,   0,   0,   0],
       [  2,   0,   0, ...,   0,   0,   0],
       ...,
       [997,   0,   0, ...,   0,   0,   0],
       [998,   0,   0, ...,   0,   0,   0],
       [999,   0,   0, ...,   0,   0,   0]], dtype=int32)